# voice-generator-v3 — Colab 학습 노트북

음성 클로닝 방어 PerturbationGenerator를 Colab(T4 GPU)에서 학습한다.

**실행 전 준비**
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. wav 데이터(`원천데이터` 폴더)를 Google Drive에 업로드
   - 권장 위치: `MyDrive/New_Sample/원천데이터/`

그다음 위에서부터 셀을 순서대로 실행하면 된다.

## 1. GPU 확인
`CUDA: True` 가 나와야 한다. False면 런타임 유형을 GPU로 바꾼다.

In [ ]:
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
!nvidia-smi -L

## 2. Google Drive 마운트
팝업이 뜨면 계정 선택 후 권한을 허용한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 코드 가져오기
private repo이므로 GitHub 토큰이 필요하다. 실행하면 입력칸이 뜬다.

> 토큰이 번거로우면 이 셀 대신, 코드 폴더를 Drive에 올린 뒤
> `!cp -r "/content/drive/MyDrive/voice_generator_v3" /content/voice_generator` 로 복사해도 된다.

In [ ]:
import os, shutil
from getpass import getpass

if os.path.isdir('/content/voice_generator'):
    shutil.rmtree('/content/voice_generator')

token = getpass('GitHub 토큰 붙여넣기: ')
!git clone https://{token}@github.com/Yena07/voice-generator-v3.git /content/voice_generator
del token
!ls /content/voice_generator

## 4. 의존성 설치
torch / torchaudio / scipy / numpy / matplotlib 은 Colab 기본 제공.

In [ ]:
!pip install -q onnx onnx2torch soundfile

## 5. CAM++ 모델(campplus.onnx) 다운로드
모든 CosyVoice 버전에서 동일한 파일(약 28MB). Drive에 한 번 받아두면 재사용된다.

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/voice_pgd', exist_ok=True)
campplus_path = '/content/drive/MyDrive/voice_pgd/campplus.onnx'
if not os.path.isfile(campplus_path) or os.path.getsize(campplus_path) < 1_000_000:
    !wget -q -O "{campplus_path}" https://huggingface.co/model-scope/CosyVoice-300M/resolve/main/campplus.onnx
!ls -lh "{campplus_path}"   # 28MB 안팎이면 정상 (수십 KB면 다운로드 실패)

## 6. 경로 / 학습 설정
본인 Drive 구조에 맞게 **이 셀만** 수정한다. (데이터 폴더 이름이 다르면 `DATA_DIR` 변경)

In [ ]:
import glob

CODE_DIR   = '/content/voice_generator'
DATA_DIR   = '/content/drive/MyDrive/New_Sample/원천데이터'        # wav 루트 (재귀 탐색)
CAMPPLUS   = '/content/drive/MyDrive/voice_pgd/campplus.onnx'
OUTPUT_DIR = '/content/drive/MyDrive/voice_gen_ckpt'              # 체크포인트 저장 (Drive)

EPOCHS = 30
BATCH  = 16     # OOM 나면 8로

# 점검: 경로가 맞는지 미리 확인
assert os.path.isdir(DATA_DIR), f'데이터 폴더 없음: {DATA_DIR}'
assert os.path.isfile(CAMPPLUS), f'campplus 없음: {CAMPPLUS}'
n_wav = len(glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True))
print(f'wav {n_wav}개 발견  /  output: {OUTPUT_DIR}')
assert n_wav > 0, 'wav가 없습니다. DATA_DIR을 확인하세요.'

## 7. 학습 시작 🚀
- 첫 실행은 원본 임베딩 캐싱(2,000개 CAM++ forward, 몇 분) 후 학습이 시작된다.
- epoch마다 `dist`(방어 효과, 클수록 좋음)가 출력되고, 체크포인트가 Drive에 저장된다.

In [ ]:
!python "{CODE_DIR}/run_colab.py" \
    --data_dir "{DATA_DIR}" \
    --campplus "{CAMPPLUS}" \
    --output_dir "{OUTPUT_DIR}" \
    --epochs {EPOCHS} --batch {BATCH}

## 8. (런타임이 끊겼을 때) 이어서 학습
셀 1~2(GPU 확인·Drive 마운트)와 셀 6(설정)을 다시 실행한 뒤 이 셀을 실행한다.
체크포인트가 Drive에 매 epoch 저장되므로 그대로 재개된다.

In [ ]:
!python "{CODE_DIR}/run_colab.py" --resume \
    --data_dir "{DATA_DIR}" \
    --campplus "{CAMPPLUS}" \
    --output_dir "{OUTPUT_DIR}" \
    --epochs {EPOCHS} --batch {BATCH}

## 9. 학습 결과 확인 & 곡선 보기

In [ ]:
!ls -lh "{OUTPUT_DIR}"
from IPython.display import Image, display
import os
png = os.path.join(OUTPUT_DIR, 'train_curves.png')
if os.path.isfile(png):
    display(Image(png))

## 10. 학습된 모델로 보호 적용 (추론)
임의의 wav 한 개에 보호 noise를 입혀 `*_protected.wav` 로 저장한다.

In [ ]:
INPUT_WAV  = '/content/drive/MyDrive/New_Sample/원천데이터'   # ← 보호할 wav 파일 경로로 바꾸기
# 예시: 데이터에서 첫 wav 하나 자동 선택
import glob, os
if os.path.isdir(INPUT_WAV):
    INPUT_WAV = glob.glob(os.path.join(INPUT_WAV, '**', '*.wav'), recursive=True)[0]
print('input:', INPUT_WAV)

!python "{CODE_DIR}/infer.py" \
    --checkpoint "{OUTPUT_DIR}/best.pt" \
    --input "{INPUT_WAV}" \
    --output "/content/protected.wav"

In [ ]:
# 원본 vs 보호본 들어보기
from IPython.display import Audio, display
print('원본'); display(Audio(INPUT_WAV))
print('보호본'); display(Audio('/content/protected.wav'))